# Dual-Branch AI Image Detector
This notebook implements the spatial and frequency based detector described in your paper.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms
from torch.utils.data import DataLoader, Dataset, random_split
from torchvision.datasets import ImageFolder
import torch.fft
import numpy as np

# Ensure reproducibility
torch.manual_seed(42)
np.random.seed(42)


In [2]:
# 1. Spatial Branch (CNN)
class SpatialBranch(nn.Module):
    def __init__(self, backbone_name='efficientnet_b4'):
        super(SpatialBranch, self).__init__()
        
        if backbone_name == 'efficientnet_b4':
            # Pretrained EfficientNet-B4 as mentioned in the paper
            self.backbone = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.DEFAULT)
            # Remove the final classification layer to extract features
            self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])
            self.feature_dim = 1792  # Output feature dimension for EfficientNet-B4
            self._backbone_type = 'standard'
            
        elif backbone_name == 'resnet50':
            self.backbone = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
            self.feature_extractor = nn.Sequential(*list(self.backbone.children())[:-1])
            self.feature_dim = 2048
            self._backbone_type = 'standard'
            
        elif backbone_name == 'convnext_base':
            # State-of-the-Art CNN -- ConvNeXt-Base
            # ConvNeXt adopts Transformer-style macro design (patchify stem,
            # depthwise conv, inverted bottleneck) while remaining a pure CNN.
            # Architecture: features (7 stages) -> avgpool -> classifier
            # We extract features + avgpool and discard the classifier head.
            _backbone = models.convnext_base(weights=models.ConvNeXt_Base_Weights.IMAGENET1K_V1)
            self.feature_extractor = _backbone.features  # 7-stage ConvNeXt blocks
            self.avgpool = _backbone.avgpool             # AdaptiveAvgPool2d
            self.feature_dim = 1024                      # ConvNeXt-Base output channels
            self._backbone_type = 'convnext'
            
        elif backbone_name == 'convnext_small':
            _backbone = models.convnext_small(weights=models.ConvNeXt_Small_Weights.IMAGENET1K_V1)
            self.feature_extractor = _backbone.features
            self.avgpool = _backbone.avgpool
            self.feature_dim = 768
            self._backbone_type = 'convnext'
            
        elif backbone_name == 'convnext_tiny':
            _backbone = models.convnext_tiny(weights=models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
            self.feature_extractor = _backbone.features
            self.avgpool = _backbone.avgpool
            self.feature_dim = 768
            self._backbone_type = 'convnext'
            
        else:
            raise ValueError(
                f"Unknown backbone: '{backbone_name}'. "
                f"Choose from: efficientnet_b4, resnet50, "
                f"convnext_base, convnext_small, convnext_tiny"
            )
            
        # Approximation of Artifact Purification Network (APN) orthogonal decomposition
        # Separates features into artifact-related and content subspaces
        self.artifact_attention = nn.Sequential(
            nn.Linear(self.feature_dim, self.feature_dim // 4),
            nn.ReLU(),
            nn.Linear(self.feature_dim // 4, self.feature_dim),
            nn.Sigmoid()
        )
        
        print(f"[SpatialBranch] Backbone: {backbone_name} | feature_dim: {self.feature_dim}")
        
    def forward(self, x):
        # Extract features -- two paths depending on backbone family
        if self._backbone_type == 'convnext':
            # ConvNeXt: features (7-stage) -> avgpool -> flatten
            x = self.feature_extractor(x)  # (B, 1024, H', W')
            x = self.avgpool(x)            # (B, 1024, 1, 1)
            features = x.flatten(1)        # (B, 1024)
        else:
            # EfficientNet / ResNet: sequential feature extractor -> flatten
            features = self.feature_extractor(x)
            features = features.view(features.size(0), -1)  # (B, feature_dim)
        
        # Apply purification (attention) to extract forgery signals
        attention_weights = self.artifact_attention(features)
        purified_features = features * attention_weights
        
        return purified_features


In [3]:
# 2. Frequency Branch (2D-DFT)
class FrequencyBranch(nn.Module):
    def __init__(self):
        super(FrequencyBranch, self).__init__()
        # Frequency band proposal mechanism
        # Uses a lightweight CNN to analyze the magnitude spectrum and find specific forgery frequency bands
        self.freq_analyzer = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(16, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.feature_dim = 32

    def forward(self, x):
        # Apply 2D-DFT
        # x is of shape (Batch, Channels, Height, Width)
        fft_result = torch.fft.fft2(x)
        # Shift the zero-frequency component to the center of the spectrum
        fft_shifted = torch.fft.fftshift(fft_result)
        
        # Extract magnitude spectrum
        # Using log scale to compress dynamic range
        magnitude = torch.log(torch.abs(fft_shifted) + 1e-8)
        
        # Pass through the frequency analyzer
        freq_features = self.freq_analyzer(magnitude)
        freq_features = freq_features.view(freq_features.size(0), -1)
        return freq_features


In [4]:
# 3. Dual-Branch Detector and Fusion
class DualBranchDetector(nn.Module):
    def __init__(self, backbone_name='efficientnet_b4'):
        super(DualBranchDetector, self).__init__()
        self.spatial_branch = SpatialBranch(backbone_name)
        self.frequency_branch = FrequencyBranch()
        
        # Fusion and Classifier
        # Combines the spatial and frequency features
        fusion_dim = self.spatial_branch.feature_dim + self.frequency_branch.feature_dim
        
        self.classifier = nn.Sequential(
            nn.Linear(fusion_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 1)
            # Sigmoid is applied via BCEWithLogitsLoss during training
        )
        
    def forward(self, x):
        # Extract features from both branches
        spatial_feat = self.spatial_branch(x)
        freq_feat = self.frequency_branch(x)
        
        # Fuse spatial and frequency features
        fused = torch.cat((spatial_feat, freq_feat), dim=1)
        
        # Final prediction
        out = self.classifier(fused)
        return out


In [5]:
# 4. Data Preparation
from torch.utils.data import Subset
import numpy as np

def get_dataloaders(data_dir, batch_size=32, subset_per_class=500):
    # Table I Specifications: Input image size downsized to 224x224
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    
    # Load the full dataset (Lazy loading, doesn't put all images in RAM yet)
    full_dataset = ImageFolder(root=data_dir, transform=transform)
    
    # --- SUBSET EXTRACTION LOGIC ---
    print(f"Original dataset size: {len(full_dataset)}")
    print(f"Classes found: {full_dataset.classes}")
    
    # Get all the labels/targets
    targets = np.array(full_dataset.targets)
    
    # Separate indices by class (Class 0 and Class 1)
    class_0_indices = np.where(targets == 0)[0]
    class_1_indices = np.where(targets == 1)[0]
    
    # Randomly select 'subset_per_class' images from each
    np.random.seed(42) # For reproducibility
    sampled_class_0 = np.random.choice(class_0_indices, subset_per_class, replace=False)
    sampled_class_1 = np.random.choice(class_1_indices, subset_per_class, replace=False)
    
    # Combine them and shuffle
    subset_indices = np.concatenate([sampled_class_0, sampled_class_1])
    np.random.shuffle(subset_indices)
    
    # Create the smaller dataset
    subset_dataset = Subset(full_dataset, subset_indices)
    print(f"New Subset size: {len(subset_dataset)} images (50% real, 50% AI)")
    
    # --- SPLITTING LOGIC ---
    # Table I Specifications: 70% Train, 15% Validation, 15% Test Split
    total_size = len(subset_dataset)
    train_size = int(0.7 * total_size)
    val_size = int(0.15 * total_size)
    test_size = total_size - train_size - val_size
    
    # Random seed for consistency
    generator = torch.Generator().manual_seed(42)
    train_dataset, val_dataset, test_dataset = random_split(
        subset_dataset, [train_size, val_size, test_size], generator=generator
    )
    
    # Create dataloaders
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)
    
    print(f"Dataset Split: {train_size} Train | {val_size} Val | {test_size} Test")
    return train_loader, val_loader, test_loader


In [6]:
# 5. Training Loop
def train_model(model, train_loader, val_loader, epochs=50, lr=1e-4, patience=5, device='cuda'):
    model = model.to(device)
    
    # Table I Specifications: Optimizer Adam, Learning Rate 1x10^-4
    optimizer = optim.Adam(model.parameters(), lr=lr)
    # Loss Function: Binary Cross-Entropy
    criterion = nn.BCEWithLogitsLoss()
    
    # Learning rate scheduling: ReduceLROnPlateau, factor of 0.5, patience 3
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
    
    best_val_loss = float('inf')
    epochs_no_improve = 0
    
    for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        train_loss = 0.0
        for inputs, labels in train_loader:
            inputs = inputs.to(device)
            # Reshape labels for BCEWithLogitsLoss
            labels = labels.to(device).float().unsqueeze(1)
            
            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            
        train_loss = train_loss / len(train_loader.dataset)
        
        # --- Validation Phase ---
        model.eval()
        val_loss = 0.0
        val_corrects = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs = inputs.to(device)
                labels = labels.to(device).float().unsqueeze(1)
                
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                val_loss += loss.item() * inputs.size(0)
                
                # Apply sigmoid and threshold to get predictions
                preds = torch.sigmoid(outputs) >= 0.5
                val_corrects += torch.sum(preds == labels.data)
                
        val_loss = val_loss / len(val_loader.dataset)
        val_acc = val_corrects.double() / len(val_loader.dataset)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        
        # Step the scheduler based on validation loss
        scheduler.step(val_loss)
        
        # Early Stopping logic
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            epochs_no_improve = 0
            # Save the best model weights
            torch.save(model.state_dict(), 'best_detector_model.pth')
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= patience:
                print(f"Early stopping triggered after {patience} epochs without improvement.")
                break


In [7]:
# 6. Execution Setup

# Set the device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Path to your GenImage / Glide dataset directory
DATA_DIR = r"C:\Users\LOQ\Documents\Datasets_Thesis\GLide\imagenet_glide\train"

# Initialize DataLoaders (Uncomment to run)
train_loader, val_loader, test_loader = get_dataloaders(DATA_DIR, batch_size=8)

# Initialize Model (EfficientNet-B4 by default) (Uncomment to run)
model = DualBranchDetector(backbone_name='convnext_base')

# Start Training (Uncomment to run)
train_model(model, train_loader, val_loader, epochs=50, lr=1e-4, patience=5, device=device)


Using device: cuda
Original dataset size: 324000
Classes found: ['ai', 'nature']
New Subset size: 1000 images (50% real, 50% AI)
Dataset Split: 700 Train | 150 Val | 150 Test
Epoch 1/50 | Train Loss: 0.6858 | Val Loss: 0.6824 | Val Acc: 0.6467
Epoch 2/50 | Train Loss: 0.6001 | Val Loss: 0.5797 | Val Acc: 0.7333
Epoch 3/50 | Train Loss: 0.2943 | Val Loss: 0.3938 | Val Acc: 0.8067
Epoch 4/50 | Train Loss: 0.1956 | Val Loss: 0.4114 | Val Acc: 0.8200
Epoch 5/50 | Train Loss: 0.1353 | Val Loss: 0.3793 | Val Acc: 0.8267
Epoch 6/50 | Train Loss: 0.1269 | Val Loss: 0.3357 | Val Acc: 0.8600
Epoch 7/50 | Train Loss: 0.0993 | Val Loss: 0.4362 | Val Acc: 0.8133
Epoch 8/50 | Train Loss: 0.0678 | Val Loss: 0.3415 | Val Acc: 0.8467
Epoch 9/50 | Train Loss: 0.0713 | Val Loss: 0.3433 | Val Acc: 0.8400
Epoch 10/50 | Train Loss: 0.0470 | Val Loss: 0.4164 | Val Acc: 0.8267
Epoch 11/50 | Train Loss: 0.0339 | Val Loss: 0.2654 | Val Acc: 0.8600
Epoch 12/50 | Train Loss: 0.0237 | Val Loss: 0.2667 | Val Acc: 0

In [9]:
# 7. Model Evaluation on Test Set
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(model, test_loader, device='cuda'):
    # Load the best weights saved during training
    model.load_state_dict(torch.load('best_detector_model.pth'))
    model = model.to(device)
    model.eval()
    
    all_preds = []
    all_labels = []
    
    print("Evaluating on Test Set...")
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device).float() # Make sure it matches prediction shape
            
            outputs = model(inputs)
            # Apply sigmoid and convert to 0 or 1
            preds = (torch.sigmoid(outputs).squeeze() >= 0.5).int()
            
            # Store predictions and true labels
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    # Calculate paper metrics
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds)
    rec = recall_score(all_labels, all_preds)
    f1 = f1_score(all_labels, all_preds)
    
    print("\n--- Final Test Metrics ---")
    print(f"Accuracy (Overall): {acc*100:.2f}%")
    print(f"Precision: {prec*100:.2f}%")
    print(f"Recall (Fake Acc / F.Acc.): {rec*100:.2f}%")
    print(f"F1-Score: {f1*100:.2f}%")

# Run the evaluation!
evaluate_model(model, test_loader, device=device)


C:\Users\LOQ\AppData\Local\Temp\ipykernel_25020\3827965552.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('best_detector_model.pth'))


Evaluating on Test Set...

--- Final Test Metrics ---
Accuracy (Overall): 90.67%
Precision: 98.21%
Recall (Fake Acc / F.Acc.): 80.88%
F1-Score: 88.71%
